In [1]:
import pandas as pd
import os

### Files

See **parameter explanation for detail information

In [2]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [3]:
files

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

In [4]:
shared_path = "../Data/R3Data"

### 1. pathology 
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [5]:
idx = 1
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'SIDE']


In [7]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['pathology.txt', 'pathology.csv', 'pathology.csv']

### Cancer

In [8]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [10]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

file not exist in: R3_3787_Lee_Data_Cancer_20240509
✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [11]:
final_df

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,SITE_NUMBER,SIDE,LESION_CLASS,LESION_LOCATION,TECHNIQUE,COMMENTS
0,4335732771,473763,04/29/2020,744032,NaN,R,Benign,Central,Surgical,NaN
1,4334946548,489405,03/18/2020,44478600,NaN,R,Benign,3 o'clock,Ultrasound guided,DR CRAIG PATIENT
2,4333066818,480693,02/04/2020,8380502,NaN,L,Malignant,1 o'clock,Ultrasound guided,"clinical stage 3A (T3, N1, M0) post neo stage ..."
3,4333541745,480240,01/07/2020,8000235,NaN,L,Malignant,Central,Surgical,NaN
4,4334476644,473646,10/05/2020,1288379,NaN,R,Malignant,Central,Surgical,NaN
...,...,...,...,...,...,...,...,...,...,...
11317,4333706909,352510,11/04/2022,1272298,1.0,L,Malignant,2 o'clock,Vacuum assisted,NaN
11318,4335176787,352269,11/18/2022,1289913,2.0,R,Benign,3 o'clock,Vacuum assisted,NaN
11319,4335969888,357236,12/07/2022,44221241,1.0,R,Malignant,3 o'clock,Ultrasound guided,NaN
11320,4333205138,351273,11/03/2022,1181952,1.0,R,Benign,Axilla,Ultrasound guided,no evidence of metastatic disease


In [12]:
final_df['PATHOLOGY_DATE'] = pd.to_datetime(final_df['PATHOLOGY_DATE'], format='%m/%d/%Y')
final_df['PATHOLOGY_DATE'] = final_df['PATHOLOGY_DATE'].dt.strftime('%Y-%m-%d')

In [13]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, True, False], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [14]:
unique_df = final_df.copy()

In [15]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [16]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [17]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [18]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,SITE_NUMBER,SIDE,LESION_CLASS,LESION_LOCATION,TECHNIQUE,COMMENTS,duplicate_count
0,4330018595,487690,2020-06-05,9654707,1.0,R,Malignant,10 o'clock,Ultrasound guided,NaN,1
1,4330018595,475789,2020-07-28,9654707,1.0,R,Malignant,10 o'clock,Ultrasound guided,NaN,1


In [19]:
unique_df['duplicate_count'].unique()

array([1])

In [24]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330018595,487690,2020-06-05,Malignant,R
1,4330018595,475789,2020-07-28,Malignant,R
2,4330029102,477464,2021-05-10,Benign,R
3,4330044371,467808,2022-05-23,Benign,R
4,4330066079,486467,2020-06-17,Malignant,L
...,...,...,...,...,...
11317,4339933706,449749,2017-07-25,Benign,L
11318,4339943307,410232,2019-06-25,Benign,R
11319,4339945522,353888,2022-10-25,High Risk,L
11320,4339945522,350178,2022-12-15,High Risk,L


In [25]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)

### Control

In [30]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['pathology_controls.txt', 'pathology_controls.csv']

In [31]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [32]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [33]:
final_df

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,SITE_NUMBER,SIDE,LESION_CLASS,LESION_LOCATION,TECHNIQUE,COMMENTS
0,4333373719,471839,10/29/2020,8400344,NaN,R,Benign,12 o'clock,Ultrasound guided,NaN
1,4333062366,475428,08/06/2020,2421647,1.0,L,Benign,6 o'clock,Stereotactic,NaN
2,4333062366,471969,12/04/2020,2421647,1.0,L,Benign,6 o'clock,MRI guided,NaN
3,4333373719,471830,10/29/2020,8400344,1.0,R,Benign,12 o'clock,Ultrasound guided,NaN
4,4334928638,426146,01/02/2019,44319710,1.0,L,Benign,NaN,Stereotactic,NaN
...,...,...,...,...,...,...,...,...,...,...
564,4334266518,351029,12/12/2022,8048926,1.0,L,Benign,12 o'clock,Ultrasound guided,NaN
565,4334560290,352629,12/14/2022,6541491,1.0,R,Benign,10 o'clock,Stereotactic,NaN
566,4333422035,351583,12/15/2022,653324,NaN,R,Benign,9 o'clock,Ultrasound guided,18 g needle lateral approach 6 cc light brown ...
567,4333422035,351381,12/22/2022,653324,1.0,R,Benign,9 o'clock,Ultrasound guided,NaN


In [35]:
final_df['PATHOLOGY_DATE'] = pd.to_datetime(final_df['PATHOLOGY_DATE'], format='%m/%d/%Y')
final_df['PATHOLOGY_DATE'] = final_df['PATHOLOGY_DATE'].dt.strftime('%Y-%m-%d')

In [36]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', "BX_ID"], ascending=[True, True, False], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [37]:
unique_df = final_df.copy()

In [38]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [39]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [40]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [42]:
unique_df['duplicate_count'].unique()

array([1])

In [43]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,duplicate_count
0,4330189702,418851,2019-04-26,Malignant,R,1
1,4330311096,450823,2016-11-10,Malignant,R,1
2,4330311096,459526,2016-11-28,Malignant,R,1
3,4330329490,437485,2017-10-25,Benign,L,1
4,4330329490,425877,2017-11-10,Benign,L,1
...,...,...,...,...,...,...
564,4337635030,474706,2019-12-30,Benign,L,1
565,4337636503,406140,2015-03-09,Benign,L,1
566,4337689949,420192,2018-08-02,Benign,L,1
567,4337793359,414656,2019-01-18,Malignant,R,1


In [44]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)